# 🛍️ Black Friday Sales Insights
## Data Mining Summative Assessment — Scenario 1
### Course: Data Mining | CRS: Artificial Intelligence
---

## 📦 Install & Import Libraries

In [ ]:
# Install required libraries (run once)
# !pip install pandas numpy matplotlib seaborn scikit-learn mlxtend plotly scipy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from scipy import stats
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('✅ All libraries imported successfully!')

---
## 🎯 Stage 1: Define the Project Scope

**Scenario:** We are Data Analysts at *Insight Mart Analytics*, working with a large retail chain during their Black Friday mega sale.

**Dataset Columns:**
- `User_ID` — Unique customer identifier
- `Product_ID` — Unique product identifier
- `Gender` — M/F
- `Age` — Age group of the customer
- `Occupation` — Occupation code (0–20)
- `City_Category` — City tier (A, B, C)
- `Stay_In_Current_City_Years` — Years living in current city
- `Marital_Status` — 0 = Single, 1 = Married
- `Product_Category_1/2/3` — Product category codes
- `Purchase` — Purchase amount in USD

**Objectives:**
1. Identify shopping behaviors (who buys what and how much)
2. Cluster customers into distinct segments
3. Find frequently co-purchased product categories (association rules)
4. Detect anomalous (unusually high) spenders
5. Deploy insights via Streamlit dashboard

---
## 🧹 Stage 2: Data Cleaning & Preprocessing

In [ ]:
# ── Load Dataset ──────────────────────────────────────────────────────────
# Download from: https://drive.google.com/drive/folders/13DxtCVj3S_AAYXG5THw2mmr6_VA1N3L9
# Place the CSV in the same folder as this notebook, then update the filename below:

df = pd.read_csv('BlackFriday.csv')   # ← update filename if needed
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# ── Basic Info ────────────────────────────────────────────────────────────
print('=== Dataset Info ===')
df.info()
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Statistical Summary ===')
df.describe()

In [ ]:
# ── Handle Missing Values ─────────────────────────────────────────────────
df['Product_Category_2'] = df['Product_Category_2'].fillna(0).astype(int)
df['Product_Category_3'] = df['Product_Category_3'].fillna(0).astype(int)
print(f'Missing values after fill: {df.isnull().sum().sum()}')

In [ ]:
# ── Remove Duplicates ─────────────────────────────────────────────────────
before = len(df)
df = df.drop_duplicates()
print(f'Removed {before - len(df)} duplicate rows. Remaining: {len(df)}')

In [ ]:
# ── Encode Categorical Variables ──────────────────────────────────────────

# Gender: Male=0, Female=1
df['Gender_Enc'] = df['Gender'].map({'M': 0, 'F': 1})

# Age groups → ordered numbers
age_map = {'0-17': 1, '18-25': 2, '26-35': 3, '36-45': 4, '46-50': 5, '51-55': 6, '55+': 7}
df['Age_Enc'] = df['Age'].map(age_map)

# City Category
df['City_Enc'] = LabelEncoder().fit_transform(df['City_Category'])

# Stay in Current City
df['Stay_Enc'] = df['Stay_In_Current_City_Years'].replace('4+', 4).astype(int)

print('Encoding complete!')
df[['Gender', 'Gender_Enc', 'Age', 'Age_Enc', 'City_Category', 'City_Enc']].head()

In [ ]:
# ── Normalize Purchase ────────────────────────────────────────────────────
scaler = StandardScaler()
df['Purchase_Norm'] = scaler.fit_transform(df[['Purchase']])

print(f'Purchase range before: {df["Purchase"].min()} - {df["Purchase"].max()}')
print(f'Purchase range after normalization: {df["Purchase_Norm"].min():.2f} - {df["Purchase_Norm"].max():.2f}')

---
## 📊 Stage 3: Exploratory Data Analysis (EDA)

In [ ]:
# ── Purchase Distribution (Histogram) ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Purchase'], bins=50, color='#FF4B4B', edgecolor='white', alpha=0.85)
axes[0].set_title('Purchase Amount Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Purchase ($)')
axes[0].set_ylabel('Frequency')

axes[1].boxplot([df[df['Gender']=='M']['Purchase'], df[df['Gender']=='F']['Purchase']],
                labels=['Male', 'Female'], patch_artist=True,
                boxprops=dict(facecolor='#FF4B4B', alpha=0.7))
axes[1].set_title('Purchase Distribution by Gender', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Purchase ($)')

plt.tight_layout()
plt.savefig('plot_purchase_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Avg Purchase by Age Group ─────────────────────────────────────────────
age_order = ['0-17','18-25','26-35','36-45','46-50','51-55','55+']
age_purchase = df.groupby('Age')['Purchase'].mean().reindex(age_order)

plt.figure(figsize=(10, 5))
bars = plt.bar(age_purchase.index, age_purchase.values, color='#FF4B4B', alpha=0.85, edgecolor='white')
plt.title('Average Purchase Amount by Age Group', fontsize=14, fontweight='bold')
plt.xlabel('Age Group')
plt.ylabel('Avg Purchase ($)')
for bar, val in zip(bars, age_purchase.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, f'${val:,.0f}',
             ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('plot_age_purchase.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Top Product Categories ────────────────────────────────────────────────
cat_counts = df.groupby('Product_Category_1')['Purchase'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 5))
bars = plt.bar(cat_counts.index.astype(str), cat_counts.values, color='#FF4B4B', alpha=0.85, edgecolor='white')
plt.title('Total Sales by Product Category (Top 10)', fontsize=14, fontweight='bold')
plt.xlabel('Product Category')
plt.ylabel('Total Purchase ($)')
plt.tight_layout()
plt.savefig('plot_category_sales.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Scatter Plot: Purchase vs Occupation ──────────────────────────────────
plt.figure(figsize=(12, 5))
plt.scatter(df['Occupation'], df['Purchase'], alpha=0.15, color='#FF4B4B', s=10)
plt.title('Purchase vs Occupation', fontsize=14, fontweight='bold')
plt.xlabel('Occupation Code')
plt.ylabel('Purchase ($)')
plt.tight_layout()
plt.savefig('plot_scatter_occupation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Correlation Heatmap ───────────────────────────────────────────────────
corr_cols = ['Age_Enc','Gender_Enc','Occupation','Marital_Status','Stay_Enc',
             'Product_Category_1','Product_Category_2','Product_Category_3','Purchase']
corr = df[corr_cols].corr()

plt.figure(figsize=(11, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='Reds', linewidths=0.5,
            annot_kws={'size': 9})
plt.title('Correlation Heatmap of Key Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🔵 Stage 4: Clustering Analysis

In [ ]:
# ── Prepare Features ──────────────────────────────────────────────────────
features = ['Age_Enc', 'Gender_Enc', 'Occupation', 'Marital_Status', 'Purchase_Norm']
X = df[features].dropna()
print(f'Clustering on {len(X)} samples with {len(features)} features')

In [ ]:
# ── Elbow Method ──────────────────────────────────────────────────────────
inertias = []
K_range = range(2, 10)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias.append(km.inertia_)

plt.figure(figsize=(9, 4))
plt.plot(list(K_range), inertias, 'ro-', linewidth=2, markersize=8)
plt.axvline(x=4, color='blue', linestyle='--', label='Chosen K=4')
plt.title('Elbow Method — Optimal Number of Clusters', fontsize=14, fontweight='bold')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.legend()
plt.tight_layout()
plt.savefig('plot_elbow.png', dpi=150, bbox_inches='tight')
plt.show()
print('Optimal K = 4 (based on elbow point)')

In [ ]:
# ── Apply K-Means with K=4 ────────────────────────────────────────────────
K = 4
km = KMeans(n_clusters=K, random_state=42, n_init=10)
df.loc[X.index, 'Cluster'] = km.fit_predict(X)
df['Cluster'] = df['Cluster'].fillna(-1).astype(int)

cluster_labels = {
    0: 'Discount Lovers',
    1: 'Premium Buyers',
    2: 'Casual Shoppers',
    3: 'Frequent Buyers'
}
df['Cluster_Label'] = df['Cluster'].map(cluster_labels)

print('=== Cluster Distribution ===')
print(df['Cluster_Label'].value_counts())

In [ ]:
# ── Cluster Statistics ────────────────────────────────────────────────────
cluster_stats = df.groupby('Cluster_Label').agg(
    Count=('User_ID', 'count'),
    Avg_Purchase=('Purchase', 'mean'),
    Total_Purchase=('Purchase', 'sum'),
    Avg_Age=('Age_Enc', 'mean')
).round(2)
print(cluster_stats)

In [ ]:
# ── Visualize Clusters using PCA ──────────────────────────────────────────
pca = PCA(n_components=2, random_state=42)
pca_result = pca.fit_transform(X)

df_plot = X.copy()
df_plot['Cluster'] = km.labels_
df_plot['PCA1'] = pca_result[:, 0]
df_plot['PCA2'] = pca_result[:, 1]
df_plot['Cluster_Label'] = df_plot['Cluster'].map(cluster_labels)

colors = ['#FF4B4B', '#4B79FF', '#FFA500', '#00CC66']
plt.figure(figsize=(11, 7))
for i, (label, color) in enumerate(zip(cluster_labels.values(), colors)):
    mask = df_plot['Cluster_Label'] == label
    plt.scatter(df_plot.loc[mask, 'PCA1'], df_plot.loc[mask, 'PCA2'],
                label=label, color=color, alpha=0.4, s=8)
plt.title('Customer Clusters (PCA 2D Projection)', fontsize=14, fontweight='bold')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.legend(markerscale=4, fontsize=11)
plt.tight_layout()
plt.savefig('plot_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🔗 Stage 5: Association Rule Mining

In [ ]:
# ── Build Transaction Basket per User ─────────────────────────────────────
df_assoc = df[df['Product_Category_1'] != 0].copy()
df_assoc['PC1'] = df_assoc['Product_Category_1'].apply(lambda x: f'Cat_{int(x)}')
df_assoc['PC2'] = df_assoc['Product_Category_2'].apply(lambda x: f'Cat_{int(x)}' if x != 0 else None)

transactions = df_assoc.groupby('User_ID').apply(
    lambda x: list(set(
        list(x['PC1']) + [v for v in x['PC2'].dropna().tolist() if v != 'Cat_0']
    ))
).tolist()

print(f'Total transactions (users): {len(transactions)}')
print(f'Sample transaction: {transactions[0]}')

In [ ]:
# ── Apriori Algorithm ─────────────────────────────────────────────────────
te = TransactionEncoder()
te_arr = te.fit_transform(transactions)
df_te = pd.DataFrame(te_arr, columns=te.columns_)

frequent_itemsets = apriori(df_te, min_support=0.05, use_colnames=True)
print(f'Frequent itemsets found: {len(frequent_itemsets)}')
frequent_itemsets.sort_values('support', ascending=False).head(10)

In [ ]:
# ── Generate Association Rules ─────────────────────────────────────────────
rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1.2)
rules = rules[rules['confidence'] >= 0.3].sort_values('lift', ascending=False)

print(f'Total rules generated: {len(rules)}')

# Clean display
display_rules = rules[['antecedents','consequents','support','confidence','lift']].copy()
display_rules['antecedents'] = display_rules['antecedents'].apply(lambda x: ', '.join(list(x)))
display_rules['consequents'] = display_rules['consequents'].apply(lambda x: ', '.join(list(x)))
display_rules.head(15)

In [ ]:
# ── Visualize: Support vs Confidence ──────────────────────────────────────
plt.figure(figsize=(10, 6))
scatter = plt.scatter(rules['support'], rules['confidence'],
                      c=rules['lift'], cmap='Reds', s=rules['lift']*20, alpha=0.8)
plt.colorbar(scatter, label='Lift')
plt.title('Association Rules: Support vs Confidence (size=Lift)', fontsize=14, fontweight='bold')
plt.xlabel('Support')
plt.ylabel('Confidence')
plt.tight_layout()
plt.savefig('plot_association_rules.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🚨 Stage 6: Anomaly Detection

In [ ]:
# ── Total Purchase per User ───────────────────────────────────────────────
user_purchase = df.groupby('User_ID')['Purchase'].sum().reset_index()
user_purchase.columns = ['User_ID', 'Total_Purchase']
user_info = df[['User_ID','Gender','Age','Occupation','City_Category','Marital_Status']].drop_duplicates('User_ID')
user_data = user_purchase.merge(user_info, on='User_ID')
print(f'Total unique users: {len(user_data)}')

In [ ]:
# ── Z-Score Method ────────────────────────────────────────────────────────
z_scores = np.abs(stats.zscore(user_data['Total_Purchase']))
anomalies_z = user_data[z_scores > 3].copy()
anomalies_z['Z_Score'] = z_scores[z_scores > 3]
print(f'Anomalies detected (Z-Score > 3): {len(anomalies_z)}')
anomalies_z.sort_values('Total_Purchase', ascending=False).head(10)

In [ ]:
# ── IQR Method ────────────────────────────────────────────────────────────
Q1 = user_data['Total_Purchase'].quantile(0.25)
Q3 = user_data['Total_Purchase'].quantile(0.75)
IQR_val = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR_val
upper_bound = Q3 + 1.5 * IQR_val

anomalies_iqr = user_data[(user_data['Total_Purchase'] < lower_bound) |
                           (user_data['Total_Purchase'] > upper_bound)]
print(f'IQR bounds: ${lower_bound:,.0f} — ${upper_bound:,.0f}')
print(f'Anomalies detected (IQR method): {len(anomalies_iqr)}')

In [ ]:
# ── Visualize Anomalies ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram with anomalies
normal = user_data[~user_data['User_ID'].isin(anomalies_z['User_ID'])]
axes[0].hist(normal['Total_Purchase'], bins=50, color='#4B79FF', alpha=0.7, label='Normal')
axes[0].hist(anomalies_z['Total_Purchase'], bins=10, color='#FF4B4B', alpha=0.9, label='Anomaly')
axes[0].axvline(x=upper_bound, color='yellow', linestyle='--', label=f'IQR Upper ({upper_bound:,.0f})')
axes[0].set_title('Purchase Distribution: Normal vs Anomalous', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Total Purchase ($)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Box Plot
axes[1].boxplot(user_data['Total_Purchase'], patch_artist=True,
                boxprops=dict(facecolor='#FF4B4B', alpha=0.7),
                flierprops=dict(marker='o', color='red', markersize=4))
axes[1].set_title('Box Plot of Total Purchase per User', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Total Purchase ($)')

plt.tight_layout()
plt.savefig('plot_anomalies.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Anomaly Demographics ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

anomalies_z['Gender'].value_counts().plot.pie(ax=axes[0], colors=['#4B79FF','#FF4B4B'],
                                              autopct='%1.1f%%', startangle=90)
axes[0].set_title('Anomalous Spenders by Gender', fontweight='bold')
axes[0].set_ylabel('')

anomalies_z['Age'].value_counts().plot.bar(ax=axes[1], color='#FF4B4B', alpha=0.85, edgecolor='white')
axes[1].set_title('Anomalous Spenders by Age Group', fontweight='bold')
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('plot_anomaly_demographics.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 💡 Stage 7: Insights & Reporting

In [ ]:
print('=' * 60)
print('        BLACK FRIDAY INSIGHTS SUMMARY REPORT')
print('=' * 60)

# Insight 1: Top spending age group
top_age = df.groupby('Age')['Purchase'].mean().idxmax()
top_age_val = df.groupby('Age')['Purchase'].mean().max()
print(f'\n📌 Insight 1 — Highest Spending Age Group')
print(f'   → Age group "{top_age}" has the highest avg spend: ${top_age_val:,.2f}')

# Insight 2: Gender
gender_avg = df.groupby('Gender')['Purchase'].mean()
print(f'\n📌 Insight 2 — Gender Spending')
print(f'   → Male avg: ${gender_avg["M"]:,.2f} | Female avg: ${gender_avg["F"]:,.2f}')
print(f'   → {"Males" if gender_avg["M"] > gender_avg["F"] else "Females"} spend more on average')

# Insight 3: Top category
top_cat = df.groupby('Product_Category_1')['Purchase'].sum().idxmax()
print(f'\n📌 Insight 3 — Most Popular Product Category')
print(f'   → Category {top_cat} generates the most total revenue')

# Insight 4: Anomalies
print(f'\n📌 Insight 4 — Anomalous Spenders')
print(f'   → {len(anomalies_z)} users detected as unusually high spenders (Z-Score method)')
print(f'   → Top spender: ${anomalies_z["Total_Purchase"].max():,.2f}')

# Insight 5: Association Rules
print(f'\n📌 Insight 5 — Association Rules')
print(f'   → {len(rules)} rules discovered with lift > 1.2')
top_rule = display_rules.iloc[0]
print(f'   → Top rule: [{top_rule["antecedents"]}] → [{top_rule["consequents"]}]')
print(f'     Support={rules.iloc[0]["support"]:.3f}, Confidence={rules.iloc[0]["confidence"]:.3f}, Lift={rules.iloc[0]["lift"]:.3f}')

print('\n' + '='*60)

---
## 🚀 Stage 8: Deployment on Streamlit Cloud

Run the Streamlit app locally:
```bash
streamlit run app.py
```

To deploy on Streamlit Cloud:
1. Push all files to your GitHub repo (`IDAI105(StudentID)-YourName`)
2. Go to https://share.streamlit.io
3. Sign in with GitHub
4. Click **New App** → Select your repo → Set main file as `app.py`
5. Click **Deploy!**
6. Copy and share the live link